# Hydropower optimisation — data contract

Both notebooks use the same input and output conventions. Set `DATA_DIR` to a directory containing `Data.csv` and `Kalam Climate Data.xlsx` (or set `CLIMATE_FILE`). A sample submission CSV is optional. Run cells in order. Install dependencies in the environment before running: `pandas numpy openpyxl scikit-learn` plus `tensorflow` for the CNN or `torch lightning pytorch-forecasting` for TFT. No notebook stores credentials or mounts a drive.

The raw measurements must contain `date_time`, `Source`, `current`, `power_factor`, `kwh`, and `v_red/v_yellow/v_blue`. Climate must contain `Date Time` and the weather columns below. `kwh` is hourly summed; other electrical fields are hourly means. Gaps in the raw measurements are retained as missing targets so they cannot silently become zero consumption. Climate is interpolated along time for the available climate interval. The supplied weather file must extend through the requested forecast period.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

DATA_DIR = Path(os.environ.get("HYDRO_DATA_DIR", "data"))
DATA_FILE = DATA_DIR / "Data.csv"
CLIMATE_FILE = DATA_DIR / "Kalam Climate Data.xlsx"
SAMPLE_FILE = DATA_DIR / "SampleSubmission.csv"  # optional; ID,kwh
OUTPUT_DIR = Path(os.environ.get("HYDRO_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
HORIZON_HOURS = 744  # 31 daily forecasts; adapt to the actual competition template
VALIDATION_HOURS = 24
CLIMATE_COLS = ["Temperature (°C)", "Dewpoint Temperature (°C)",
                "U Wind Component (m/s)", "V Wind Component (m/s)",
                "Total Precipitation (mm)", "Snowfall (mm)", "Snow Cover (%)"]
EXOG = CLIMATE_COLS + ["day_sin", "day_cos"]

def calendar_features(index):
    hour = index.hour + index.minute / 60
    return pd.DataFrame({"day_sin": np.sin(2 * np.pi * hour / 24),
                         "day_cos": np.cos(2 * np.pi * hour / 24)}, index=index)


## Load and prepare hourly data

Chunks are aggregated with sums and counts so a source/hour split across CSV chunks is combined correctly. Preserve source names as strings. An observed hour has an electrical target; missing hours stay `NaN` until each model handles them explicitly.

In [ ]:
def load_hourly_data(data_file=DATA_FILE, climate_file=CLIMATE_FILE, chunksize=500_000):
    if not data_file.exists() or not climate_file.exists():
        raise FileNotFoundError(f"Provide {data_file} and {climate_file}")
    measures = ["current", "power_factor", "kwh", "voltage"]
    parts = []
    for chunk in pd.read_csv(data_file, chunksize=chunksize, parse_dates=["date_time"]):
        needed = {"date_time", "Source", "current", "power_factor", "kwh",
                  "v_red", "v_yellow", "v_blue"}
        if missing := needed.difference(chunk.columns):
            raise ValueError(f"Missing columns in Data.csv: {sorted(missing)}")
        chunk = chunk.dropna(subset=["date_time", "Source"]).copy()
        chunk["Source"] = chunk["Source"].astype(str)
        for col in needed - {"date_time", "Source"}:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
        chunk["voltage"] = chunk[["v_red", "v_yellow", "v_blue"]].sum(axis=1, min_count=1)
        chunk["timestamp"] = chunk["date_time"].dt.ceil("h")  # right-closed hour
        grouped = chunk.groupby(["Source", "timestamp"], sort=False)
        block = grouped[measures].agg(["sum", "count"])
        block.columns = [f"{name}_{stat}" for name, stat in block.columns]
        parts.append(block)
    if not parts:
        raise ValueError("Data.csv contains no records")
    totals = pd.concat(parts).groupby(level=[0, 1]).sum()
    hourly = pd.DataFrame(index=totals.index)
    for name in measures:
        hourly[name] = (totals[f"{name}_sum"] if name == "kwh" else
                        totals[f"{name}_sum"] / totals[f"{name}_count"].replace(0, np.nan))
        hourly.loc[totals[f"{name}_count"].eq(0), name] = np.nan
    hourly = hourly.reset_index()
    climate = pd.read_excel(climate_file, parse_dates=["Date Time"])
    missing = set(CLIMATE_COLS).difference(climate.columns)
    if missing:
        raise ValueError(f"Missing climate columns: {sorted(missing)}")
    climate = climate.set_index("Date Time")[CLIMATE_COLS].sort_index()
    climate = climate.apply(pd.to_numeric, errors="coerce")
    climate = climate.groupby(level=0).mean()
    climate = climate.resample("h").mean().interpolate(method="time", limit_area="inside")
    climate = climate.join(calendar_features(climate.index))
    hourly = hourly.join(climate, on="timestamp", how="inner")
    hourly = hourly.sort_values(["Source", "timestamp"]).reset_index(drop=True)
    if hourly.empty:
        raise ValueError("Measurements and climate have no overlapping timestamps")
    return hourly, climate

hourly, climate = load_hourly_data()
print(f"{len(hourly):,} observed source-hours; {hourly.Source.nunique()} sources")


## Forecast dates and submission checks

The default starts immediately after the last measurement and forecasts 744 hours. If the official sample submission is available, its daily dates and source IDs determine the requested span and row order. Verify the template matches this project's `YYYY-MM-DD_Source` ID pattern before exporting.

In [ ]:
def forecast_request(hourly, sample_file=SAMPLE_FILE):
    sample = pd.read_csv(sample_file) if sample_file.exists() else None
    if sample is not None:
        if "ID" not in sample or sample["ID"].duplicated().any():
            raise ValueError("Sample submission needs unique IDs")
        dates = pd.to_datetime(sample["ID"].str.slice(0, 10), format="%Y-%m-%d", errors="raise")
        sources = sample["ID"].str.slice(11)
        if not (sample["ID"].str.slice(10, 11) == "_").all():
            raise ValueError("Expected sample IDs in YYYY-MM-DD_Source format")
        if sources.isna().any() or sources.eq("").any():
            raise ValueError("Sample submission has empty source names")
        expected = {f"{d.date()}_{s}" for d, s in zip(dates, sources)}
        if expected != set(sample["ID"]):
            raise ValueError("Sample IDs do not match the expected date/source format")
        start = dates.min().normalize()
        end = dates.max().normalize() + pd.Timedelta(hours=23)
        requested_sources = set(sources)
    else:
        start = hourly.timestamp.max() + pd.Timedelta(hours=1)
        if start.hour != 0:
            raise ValueError("Daily forecast must begin at midnight; supply a sample submission or adjust the forecast start")
        end = start + pd.Timedelta(hours=HORIZON_HOURS - 1)
        requested_sources = set(hourly.Source)
    if start <= hourly.timestamp.max():
        raise ValueError("Requested forecast overlaps observed training timestamps")
    if end > climate.index.max() or start < climate.index.min():
        raise ValueError("Climate file must cover the entire forecast interval")
    missing = requested_sources - set(hourly.Source)
    if missing:
        raise ValueError(f"Unknown sources in template: {sorted(missing)[:5]}")
    future_hours = pd.date_range(start, end, freq="h")
    if sample is None and len(future_hours) % 24:
        raise ValueError("Daily submission needs a multiple of 24 hours")
    return future_hours, requested_sources, sample

def make_submission(hourly_forecast, sample):
    daily = (hourly_forecast.assign(date=hourly_forecast.timestamp.dt.normalize())
             .groupby(["date", "Source"], as_index=False)["kwh"].sum())
    counts = hourly_forecast.assign(date=hourly_forecast.timestamp.dt.normalize()).groupby(
        ["date", "Source"]).size()
    if not counts.eq(24).all():
        raise ValueError("A forecast day does not have all 24 hourly predictions")
    daily["ID"] = daily.date.dt.strftime("%Y-%m-%d") + "_" + daily.Source
    result = daily[["ID", "kwh"]]
    if sample is not None:
        if set(result.ID) != set(sample.ID):
            raise ValueError("Forecast IDs do not match the sample submission")
        result = sample[["ID"]].merge(result, on="ID", validate="one_to_one")
    if not np.isfinite(result.kwh).all():
        raise ValueError("Forecast contains missing or infinite values")
    return result

future_hours, requested_sources, sample = forecast_request(hourly)
print(f"Forecast: {future_hours.min()} to {future_hours.max()}; {len(requested_sources)} sources")


# Model A — 1D CNN + LSTM

A 168 hour convolutional history and the most recent 24 hours feed a one step regression head for `power_factor`, `power` and `kwh`. A learned source embedding conditions the sequence. The final convolution uses stride 7 and **no pooling** so its 24 steps align with the LSTM input. The proposed five dense layers, 32 LSTM units, batch size 64, MinMax scaling and RMSprop are explicit below. Adam is available as a configuration choice.

Validation reserves the last 24 observed hours per source. Each source's scaling is fitted on training hours only. Samples with incomplete electrical or climate observations are skipped; the recursive forecast fails clearly if its final 168 hour history is incomplete.

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

tf.keras.utils.set_random_seed(SEED)
ENCODER_HOURS, SHORT_HOURS = 168, 24
TARGETS = ["power_factor", "power", "kwh"]
FEATURES = TARGETS + EXOG
BATCH_SIZE, EPOCHS = 64, 20
OPTIMIZER = "rmsprop"  # alternatively "adam"

source_ids = {source: i + 1 for i, source in enumerate(sorted(hourly.Source.unique()))}
scalers, series = {}, {}
for source, frame in hourly.groupby("Source"):
    frame = frame.set_index("timestamp").sort_index()
    frame = frame.reindex(pd.date_range(frame.index.min(), frame.index.max(), freq="h"))
    if len(frame) < ENCODER_HOURS + VALIDATION_HOURS + 1:
        continue
    frame["power"] = frame["current"] * frame["voltage"]
    split = len(frame) - VALIDATION_HOURS
    if frame.iloc[:split][TARGETS].dropna().empty or frame.iloc[:split][EXOG].dropna().empty:
        continue
    target_scaler = MinMaxScaler().fit(frame.iloc[:split][TARGETS].dropna())
    climate_scaler = MinMaxScaler().fit(frame.iloc[:split][EXOG].dropna())
    scaled = pd.DataFrame(index=frame.index, columns=FEATURES, dtype=float)
    # Transform only valid rows; preserving NaNs lets the window builder reject gaps.
    valid_target = frame[TARGETS].notna().all(axis=1)
    valid_exog = frame[EXOG].notna().all(axis=1)
    scaled.loc[valid_target, TARGETS] = target_scaler.transform(frame.loc[valid_target, TARGETS])
    scaled.loc[valid_exog, EXOG] = climate_scaler.transform(frame.loc[valid_exog, EXOG])
    scalers[source] = (target_scaler, climate_scaler)
    series[source] = (scaled, split)
if requested_sources - set(series):
    raise ValueError(f"Sources lack sufficient CNN training history: {sorted(requested_sources - set(series))[:5]}")
print(f"Usable CNN sources: {len(series)}")


In [ ]:
def window_examples(series, validation=False):
    for source, (frame, split) in series.items():
        values = frame[FEATURES].to_numpy(dtype=np.float32)
        targets = frame[TARGETS].to_numpy(dtype=np.float32)
        left = max(ENCODER_HOURS, split) if validation else ENCODER_HOURS
        right = len(frame) if validation else split
        for t in range(left, right):
            history, label = values[t - ENCODER_HOURS:t], targets[t]
            if not (np.isfinite(history).all() and np.isfinite(label).all()):
                continue
            yield ({"cnn_history": history,
                    "lstm_history": history[-SHORT_HOURS:],
                    "source_id": np.array([source_ids[source]], dtype=np.int32)}, label)

signature = (
    {"cnn_history": tf.TensorSpec((ENCODER_HOURS, len(FEATURES)), tf.float32),
     "lstm_history": tf.TensorSpec((SHORT_HOURS, len(FEATURES)), tf.float32),
     "source_id": tf.TensorSpec((1,), tf.int32)},
    tf.TensorSpec((len(TARGETS),), tf.float32),
)
def dataset(validation=False):
    ds = tf.data.Dataset.from_generator(lambda: window_examples(series, validation),
                                        output_signature=signature)
    if not validation:
        ds = ds.shuffle(8192, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds = dataset(), dataset(validation=True)
if not any(True for _ in window_examples(series)) or not any(True for _ in window_examples(series, True)):
    raise ValueError("No complete training or validation windows; inspect missing data")


In [ ]:
def build_cnn_lstm(n_sources, optimizer_name=OPTIMIZER):
    cnn = tf.keras.Input((ENCODER_HOURS, len(FEATURES)), name="cnn_history")
    recent = tf.keras.Input((SHORT_HOURS, len(FEATURES)), name="lstm_history")
    source = tf.keras.Input((1,), dtype="int32", name="source_id")
    x = cnn
    for filters, width, stride in [(16, 48, 1), (16, 6, 1), (32, 4, 1),
                                   (64, 4, 1), (12, 4, 7)]:
        x = tf.keras.layers.Conv1D(filters, width, strides=stride, padding="same",
                                   activation="relu")(x)
    embedded = tf.keras.layers.Embedding(n_sources + 1, 8)(source)
    embedded = tf.keras.layers.RepeatVector(SHORT_HOURS)(tf.keras.layers.Flatten()(embedded))
    x = tf.keras.layers.Concatenate(axis=-1)([x, recent, embedded])
    x = tf.keras.layers.LSTM(32)(x)
    for units in [64, 32, 16, 8, 8]:
        x = tf.keras.layers.Dense(units, activation="relu")(x)
    outputs = tf.keras.layers.Dense(len(TARGETS), name="scaled_targets")(x)
    model = tf.keras.Model({"cnn_history": cnn, "lstm_history": recent,
                            "source_id": source}, outputs)
    optimizers = {"rmsprop": tf.keras.optimizers.RMSprop,
                  "adam": tf.keras.optimizers.Adam}
    model.compile(optimizer=optimizers[optimizer_name](learning_rate=1e-3),
                  loss="mse", metrics=[tf.keras.metrics.MeanAbsoluteError()])
    return model

model = build_cnn_lstm(len(source_ids))
model.summary()


## Train, test, and reload

Validation MAE is calculated in physical `kwh` units, after inverse scaling. Compare it with a same-hour previous-day seasonal baseline. These last 24 hours are not used to fit the scalers or network. `EarlyStopping` restores the best weights, then a portable `.keras` model is saved.

In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                               restore_best_weights=True)]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
model_path = OUTPUT_DIR / "cnn_lstm.keras"
model.save(model_path)
model = tf.keras.models.load_model(model_path)

truth, predicted, seasonal = [], [], []
for source, (frame, split) in series.items():
    t_scaler = scalers[source][0]
    values = frame[FEATURES].to_numpy(dtype=np.float32)
    for t in range(max(split, ENCODER_HOURS), len(frame)):
        previous = values[t - ENCODER_HOURS:t]
        if not (np.isfinite(previous).all() and np.isfinite(values[t, 2])
                and np.isfinite(values[t - 24, 2])):
            continue
        inputs = {"cnn_history": previous[None],
                  "lstm_history": previous[-SHORT_HOURS:][None],
                  "source_id": np.array([[source_ids[source]]], dtype=np.int32)}
        pred = model(inputs, training=False).numpy()[0]
        truth.append(t_scaler.inverse_transform(values[t:t + 1, :3])[0, 2])
        predicted.append(t_scaler.inverse_transform(pred[None])[0, 2])
        seasonal.append(t_scaler.inverse_transform(values[t - 24:t - 23, :3])[0, 2])
print("Validation kwh MAE:", mean_absolute_error(truth, predicted))
print("Previous-day kwh MAE:", mean_absolute_error(truth, seasonal))


## Recursive hourly forecast

The model predicts one hour at a time. Insert that prediction alongside weather for the *predicted timestamp*, update both histories, then predict the next hour. Unknown future electrical values are supplied by the previous prediction, rather than by future ground truth.

In [ ]:
def forecast_cnn(model, sources=requested_sources, hours=future_hours):
    rows = []
    for source in sorted(sources):
        frame, _ = series[source]
        target_scaler, exog_scaler = scalers[source]
        # The same forecast start applies to all sources; require contiguous observations
        # up to the hour before the first requested prediction.
        prior = pd.date_range(end=hours[0] - pd.Timedelta(hours=1),
                              periods=ENCODER_HOURS, freq="h")
        history = frame.reindex(prior)[FEATURES].to_numpy(dtype=np.float32)
        if not np.isfinite(history).all():
            raise ValueError(f"Missing final {ENCODER_HOURS} hours for {source}")
        weather = climate.reindex(hours)[EXOG]
        if weather.isna().any().any():
            raise ValueError(f"Missing future weather for {source}")
        future_exog = exog_scaler.transform(weather).astype(np.float32)
        for stamp, weather_row in zip(hours, future_exog):
            inputs = {"cnn_history": history[None],
                      "lstm_history": history[-SHORT_HOURS:][None],
                      "source_id": np.array([[source_ids[source]]], dtype=np.int32)}
            scaled_pred = model(inputs, training=False).numpy()[0]
            actual = target_scaler.inverse_transform(scaled_pred[None])[0]
            rows.append((stamp, source, max(0.0, float(actual[2]))))
            history = np.vstack([history[1:], np.r_[scaled_pred, weather_row]])
    return pd.DataFrame(rows, columns=["timestamp", "Source", "kwh"])

hourly_forecast = forecast_cnn(model)
submission = make_submission(hourly_forecast, sample)
submission.to_csv(OUTPUT_DIR / "cnn_submission.csv", index=False)
print(submission.head())


### Notes on the recovered design

The earlier `CNN_LTSM_RELU.h5` reference referred to an unavailable checkpoint; this notebook trains, saves, and reloads its own model. The broken simultaneous 168 hour/24 hour generators and hard-coded 584 source offsets were replaced with aligned windows and a stable source index. The optional Adam optimizer implements the alternative noted in the original plan. A TFT implementation is in `hydro_optimization_TFT.ipynb`. For a fair final model, retrain on all observed history after model selection; the validation holdout above is retained for transparent comparison.